# 京都红叶日期与秋季气温关系的实证分析（2010—2025）

本研究使用日本气象厅官方物候资料与京都站日别气温数据，检验 2010—2025 年京都枫叶红叶日期与秋季气温之间的经验关系。研究目标不是建立黑箱预测模型，而是识别可解释、可复现、可用于旅行日期选择的温度信号。


## 0. 研究问题与分析框架

**研究问题：** 当京都秋季气温偏暖或偏冷时，官方枫叶红叶日期是否存在系统性提前或推迟？如果存在，这种关系对旅行日期选择有多大解释力？

分析框架如下：

1. 数据来源与获取：使用日本气象厅官方物候资料与京都站日别气温。
2. 数据清洗：统一年份范围，处理缺测记录，构造红叶日期偏移量与气温指标。
3. 统计分析：计算相关系数、线性斜率与解释度。
4. 结果可视化：展示年度偏移、11 月均温回归关系，以及不同气温指标的解释力。
5. 结论：将统计证据转化为日期选择规则，并说明适用边界。


In [1]:
from pathlib import Path
import csv
import math

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

SUMMARY = ROOT / "data" / "processed" / "kyoto_koyo_temperature_2010_2025_summary.csv"
DAILY = ROOT / "data" / "raw" / "kyoto_daily_temperature_oct_dec_2010_2025.csv"
CORRELATIONS = ROOT / "data" / "processed" / "correlation_results.csv"

def read_csv(path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

summary = read_csv(SUMMARY)
daily = read_csv(DAILY)
correlations = read_csv(CORRELATIONS)

print(f"repo root: {ROOT.name}")
print(f"summary rows: {len(summary)} years ({summary[0]['year']}-{summary[-1]['year']})")
print(f"daily rows: {len(daily)} records")
print(f"correlation metrics: {len(correlations)}")

repo root: kyoto_autumn_research
summary rows: 16 years (2010-2025)
daily rows: 1472 records
correlation metrics: 12


## 1. 数据来源与获取

红叶日期采用日本气象厅生物季节观测累年值中的京都 `かえでの紅葉日`。气温采用日本气象厅过去天气数据中的京都站日别值。

数据源链接：

- 生物季节观测累年值（かえで紅葉）：https://www.data.jma.go.jp/sakura/data/ruinenchi/015.csv
- 京都站日别天气值：https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=61&block_no=47759&year=YYYY&month=MM&day=&view=p1

为保证 Notebook 打开时具有稳定的可复现性，以下代码默认读取仓库中已经归档的 CSV 文件。如需刷新至气象厅最新数据，可在仓库根目录运行 `python3 scripts/fetch_and_analyze.py`，或将下方 `RUN_LIVE_FETCH` 改为 `True` 后执行。


In [2]:
PHENOLOGY_URL = "https://www.data.jma.go.jp/sakura/data/ruinenchi/015.csv"
WEATHER_URL_TEMPLATE = "https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=61&block_no=47759&year=YYYY&month=MM&day=&view=p1"
RUN_LIVE_FETCH = False

print("红叶日数据源:", PHENOLOGY_URL)
print("气温数据源模板:", WEATHER_URL_TEMPLATE)
print("为保证可复现性，默认读取仓库中的已生成 CSV；需要刷新时运行: python3 scripts/fetch_and_analyze.py")

if RUN_LIVE_FETCH:
    import importlib.util
    spec = importlib.util.spec_from_file_location("fetch_and_analyze", ROOT / "scripts" / "fetch_and_analyze.py")
    analysis = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(analysis)
    analysis.main()
    summary = read_csv(SUMMARY)
    daily = read_csv(DAILY)
    correlations = read_csv(CORRELATIONS)

红叶日数据源: https://www.data.jma.go.jp/sakura/data/ruinenchi/015.csv
气温数据源模板: https://www.data.jma.go.jp/stats/etrn/view/daily_s1.php?prec_no=61&block_no=47759&year=YYYY&month=MM&day=&view=p1
为保证可复现性，默认读取仓库中的已生成 CSV；需要刷新时运行: python3 scripts/fetch_and_analyze.py


## 2. 数据清洗与变量构造

清洗后的核心变量如下：

- `red_md`：官方红叶日（月/日）。
- `delay_vs_dec5_days`：相对平年日 12/5 的偏移；正数表示偏晚，负数表示偏早。
- `oct_mean_c` / `nov_mean_c` / `oct_nov_mean_c`：10 月、11 月、10—11 月平均日均温。
- `nov_dec10_days_le_10c`：11/1—12/10 期间日均温 ≤10°C 的天数。


In [3]:
def print_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(headers[i]).ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))

core_rows = []
for r in summary:
    core_rows.append([r["year"], r["red_md"] or "缺测", "" if not r["delay_vs_dec5_days"] else f"{int(r['delay_vs_dec5_days']):+d}", f"{float(r['oct_mean_c']):.2f}", f"{float(r['nov_mean_c']):.2f}", f"{float(r['oct_nov_mean_c']):.2f}", r["nov_dec10_days_le_10c"]])

print_table(core_rows, ["year", "official red", "delay d", "Oct °C", "Nov °C", "Oct-Nov °C", "≤10°C days"])

year | official red | delay d | Oct °C | Nov °C | Oct-Nov °C | ≤10°C days
-----+--------------+---------+--------+--------+------------+-----------
2010 | 12/13        | +8      | 19.11  | 11.75  | 15.49      | 13        
2011 | 12/14        | +9      | 18.39  | 13.79  | 16.13      | 11        
2012 | 12/07        | +2      | 18.22  | 11.14  | 14.74      | 20        
2013 | 12/05        | +0      | 20.05  | 11.51  | 15.85      | 22        
2014 | 12/09        | +4      | 18.75  | 13.15  | 16.00      | 13        
2015 | 12/14        | +9      | 18.11  | 14.45  | 16.31      | 11        
2016 | 12/06        | +1      | 19.70  | 12.49  | 16.15      | 12        
2017 | 12/01        | -4      | 18.04  | 11.20  | 14.68      | 21        
2018 | 12/12        | +7      | 18.72  | 13.49  | 16.15      | 5         
2019 | 12/10        | +5      | 19.97  | 12.92  | 16.50      | 14        
2020 | 12/07        | +2      | 17.91  | 13.63  | 15.80      | 12        
2021 | 缺测           |         | 19.60 

### 2.1 缺测处理与覆盖检查

2021 年京都 `かえでの紅葉` 在气象厅累年 CSV 中记录为 `0`。该值不能解释为有效日期，因此在涉及红叶日期的相关性计算中按缺测处理；同期气温记录仍保留在原始与汇总数据中。


In [4]:
coverage = {}
for r in daily:
    coverage[(r["year"], r["month"])] = coverage.get((r["year"], r["month"]), 0) + 1

coverage_bad = [k for k, v in sorted(coverage.items()) if v not in (30, 31)]
sample_2025 = next(r for r in daily if r["date"] == "2025-11-01")

print("缺测策略: 2021 年官方かえで紅葉值为 0，按缺测处理；气温数据仍保留。")
print(f"日别覆盖: {len(coverage)} 个 year-month 组合；异常组合: {coverage_bad or '无'}")
print(f"样例 2025-11-01: 平均 {sample_2025['t_mean_c']}°C, 最高 {sample_2025['t_max_c']}°C, 最低 {sample_2025['t_min_c']}°C")

缺测策略: 2021 年官方かえで紅葉值为 0，按缺测处理；气温数据仍保留。
日别覆盖: 48 个 year-month 组合；异常组合: 无
样例 2025-11-01: 平均 15.700000°C, 最高 21.300000°C, 最低 12.200000°C


## 3. 统计分析

因变量为 `delay_vs_dec5_days`。候选解释变量包括 10 月均温、11 月均温、10—11 月均温、11/1—12/10 均温，以及冷日数量。下表列出主要指标与红叶日期偏移量之间的相关性和线性斜率。


In [5]:
corr_labels = {
    "oct_nov_mean_c": "10-11月均温",
    "nov_mean_c": "11月均温",
    "nov_dec10_mean_c": "11/1-12/10均温",
    "nov_dec10_days_le_10c": "11/1-12/10 ≤10°C天数",
    "dec_mean_c": "12月均温",
    "oct_mean_c": "10月均温",
}
corr_by_metric = {r["metric"]: r for r in correlations}

corr_rows = []
for metric in ["oct_nov_mean_c", "nov_mean_c", "nov_dec10_mean_c", "nov_dec10_days_le_10c", "dec_mean_c", "oct_mean_c"]:
    r = corr_by_metric[metric]
    corr_rows.append([corr_labels[metric], r["n"], f"{float(r['pearson_r']):+.3f}", f"{float(r['slope_days_per_unit']):+.2f}", f"{float(r['r2']):.3f}"])

print_table(corr_rows, ["metric", "n", "Pearson r", "slope days/unit", "R²"])

metric             | n  | Pearson r | slope days/unit | R²   
-------------------+----+-----------+-----------------+------
10-11月均温           | 15 | +0.729    | +4.37           | 0.531
11月均温              | 15 | +0.714    | +2.98           | 0.509
11/1-12/10均温       | 15 | +0.709    | +3.01           | 0.503
11/1-12/10 ≤10°C天数 | 15 | -0.644    | -0.65           | 0.415
12月均温              | 15 | +0.393    | +1.55           | 0.155
10月均温              | 15 | +0.309    | +1.29           | 0.095


## 4. 结果可视化

以下三幅图分别展示：

1. 各年份官方红叶日期相对平年日的偏移。
2. 11 月均温与红叶日期偏移之间的线性关系。
3. 不同气温指标对红叶日期偏移的解释力比较。


In [6]:
def show_svg(svg):
    try:
        from IPython.display import HTML, display
        display(HTML(svg))
    except Exception:
        print(svg)


def scale(value, old_min, old_max, new_min, new_max):
    if old_max == old_min:
        return (new_min + new_max) / 2
    return new_min + (value - old_min) * (new_max - new_min) / (old_max - old_min)


def regression(xs, ys):
    mean_x = sum(xs) / len(xs)
    mean_y = sum(ys) / len(ys)
    ss_x = sum((x - mean_x) ** 2 for x in xs)
    slope = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys)) / ss_x
    return slope, mean_y - slope * mean_x


def svg_base(parts, width, height):
    style = "<style>.axis{stroke:#334155;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.label{font-family:Arial,sans-serif;font-size:12px;fill:#334155}.title{font-family:Arial,sans-serif;font-size:18px;font-weight:700;fill:#0f172a}.note{font-family:Arial,sans-serif;font-size:12px;fill:#64748b}</style>"
    return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">' + style + "".join(parts) + "</svg>"


def make_delay_line_svg(rows):
    pts = [(int(r["year"]), float(r["delay_vs_dec5_days"])) for r in rows if r["delay_vs_dec5_days"]]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    width, height, ml, mr, mt, mb = 820, 380, 70, 30, 55, 60
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(-5, min(ys)), max(16, max(ys))
    parts = [f'<text class="title" x="{width/2}" y="28" text-anchor="middle">官方红叶日相对 12/5 的偏移</text>']
    for y in range(int(y0), int(y1) + 1, 5):
        py = scale(y, y0, y1, height - mb, mt)
        parts.append(f'<line class="grid" x1="{ml}" y1="{py:.1f}" x2="{width-mr}" y2="{py:.1f}"/>')
        parts.append(f'<text class="label" x="{ml-10}" y="{py+4:.1f}" text-anchor="end">{y:+d}</text>')
    parts.append(f'<line class="axis" x1="{ml}" y1="{height-mb}" x2="{width-mr}" y2="{height-mb}"/>')
    parts.append(f'<line class="axis" x1="{ml}" y1="{mt}" x2="{ml}" y2="{height-mb}"/>')
    path = " ".join(f'{scale(x, x0, x1, ml, width-mr):.1f},{scale(y, y0, y1, height-mb, mt):.1f}' for x, y in pts)
    parts.append(f'<polyline fill="none" stroke="#dc2626" stroke-width="2.5" points="{path}"/>')
    for year, delay in pts:
        px = scale(year, x0, x1, ml, width - mr)
        py = scale(delay, y0, y1, height - mb, mt)
        color = "#b91c1c" if year == 2024 else "#f97316"
        parts.append(f'<circle cx="{px:.1f}" cy="{py:.1f}" r="4" fill="{color}"/>')
        if year in (2017, 2024, 2025):
            parts.append(f'<text class="label" x="{px:.1f}" y="{py-9:.1f}" text-anchor="middle">{year} {delay:+.0f}d</text>')
    for year in range(x0, x1 + 1, 3):
        px = scale(year, x0, x1, ml, width - mr)
        parts.append(f'<text class="label" x="{px:.1f}" y="{height-mb+22}" text-anchor="middle">{year}</text>')
    parts.append(f'<text class="note" x="{width/2}" y="{height-18}" text-anchor="middle">正值=比平年晚；2021 缺测未画点</text>')
    return svg_base(parts, width, height)


def make_nov_scatter_svg(rows):
    pts = [(float(r["nov_mean_c"]), float(r["delay_vs_dec5_days"]), int(r["year"])) for r in rows if r["delay_vs_dec5_days"]]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    width, height, ml, mr, mt, mb = 820, 420, 80, 40, 55, 70
    x0, x1 = min(xs) - 0.4, max(xs) + 0.4
    y0, y1 = min(-5, min(ys)), max(16, max(ys))
    slope, intercept = regression(xs, ys)
    parts = [f'<text class="title" x="{width/2}" y="28" text-anchor="middle">11月均温 vs 官方红叶延迟</text>']
    for y in range(int(y0), int(y1) + 1, 5):
        py = scale(y, y0, y1, height - mb, mt)
        parts.append(f'<line class="grid" x1="{ml}" y1="{py:.1f}" x2="{width-mr}" y2="{py:.1f}"/>')
        parts.append(f'<text class="label" x="{ml-10}" y="{py+4:.1f}" text-anchor="end">{y:+d}</text>')
    for x in [11, 12, 13, 14, 15]:
        px = scale(x, x0, x1, ml, width - mr)
        parts.append(f'<line class="grid" x1="{px:.1f}" y1="{mt}" x2="{px:.1f}" y2="{height-mb}"/>')
        parts.append(f'<text class="label" x="{px:.1f}" y="{height-mb+22}" text-anchor="middle">{x}°C</text>')
    parts.append(f'<line class="axis" x1="{ml}" y1="{height-mb}" x2="{width-mr}" y2="{height-mb}"/>')
    parts.append(f'<line class="axis" x1="{ml}" y1="{mt}" x2="{ml}" y2="{height-mb}"/>')
    trend = " ".join(f'{scale(x, x0, x1, ml, width-mr):.1f},{scale(intercept + slope*x, y0, y1, height-mb, mt):.1f}' for x in (min(xs), max(xs)))
    parts.append(f'<polyline fill="none" stroke="#2563eb" stroke-width="2.5" points="{trend}"/>')
    for x, y, year in pts:
        px = scale(x, x0, x1, ml, width - mr)
        py = scale(y, y0, y1, height - mb, mt)
        color = "#b91c1c" if year == 2024 else "#0ea5e9"
        parts.append(f'<circle cx="{px:.1f}" cy="{py:.1f}" r="5" fill="{color}" opacity="0.9"/>')
        if year in (2015, 2017, 2024, 2025):
            parts.append(f'<text class="label" x="{px+7:.1f}" y="{py-7:.1f}">{year}</text>')
    parts.append(f'<text class="note" x="{width/2}" y="{height-34}" text-anchor="middle">线性斜率约 {slope:.1f} 天/°C；Pearson r≈0.714</text>')
    parts.append(f'<text class="note" x="{width/2}" y="{height-16}" text-anchor="middle">横轴=11月均温；纵轴=相对 12/5 延迟天数</text>')
    return svg_base(parts, width, height)


def make_correlation_bar_svg(corr_by_metric):
    selected = [
        ("10-11月均温", float(corr_by_metric["oct_nov_mean_c"]["pearson_r"])),
        ("11月均温", float(corr_by_metric["nov_mean_c"]["pearson_r"])),
        ("11/1-12/10均温", float(corr_by_metric["nov_dec10_mean_c"]["pearson_r"])),
        ("≤10°C冷天数", float(corr_by_metric["nov_dec10_days_le_10c"]["pearson_r"])),
        ("12月均温", float(corr_by_metric["dec_mean_c"]["pearson_r"])),
        ("10月均温", float(corr_by_metric["oct_mean_c"]["pearson_r"])),
    ]
    width, height, ml, mr, mt, mb = 820, 390, 190, 50, 55, 40
    zero_x = scale(0, -0.8, 0.8, ml, width - mr)
    parts = [f'<text class="title" x="{width/2}" y="28" text-anchor="middle">各气温指标与红叶延迟的相关性</text>']
    parts.append(f'<line class="axis" x1="{zero_x:.1f}" y1="{mt}" x2="{zero_x:.1f}" y2="{height-mb}"/>')
    for i, (label, r) in enumerate(selected):
        y = mt + i * 48
        x = scale(min(0, r), -0.8, 0.8, ml, width - mr)
        bar_width = abs(scale(r, -0.8, 0.8, ml, width - mr) - zero_x)
        color = "#ef4444" if r > 0 else "#2563eb"
        tx = scale(r, -0.8, 0.8, ml, width - mr)
        anchor = "start" if r >= 0 else "end"
        offset = 6 if r >= 0 else -6
        parts.append(f'<text class="label" x="{ml-12}" y="{y+21}" text-anchor="end">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bar_width:.1f}" height="32" rx="4" fill="{color}" opacity="0.85"/>')
        parts.append(f'<text class="label" x="{tx+offset:.1f}" y="{y+21}" text-anchor="{anchor}">{r:+.3f}</text>')
    parts.append(f'<text class="note" x="{width/2}" y="{height-15}" text-anchor="middle">红色=越暖越晚；蓝色=冷天越多越早</text>')
    return svg_base(parts, width, height)

### 4.1 年度偏移：2024 年为样本期内最晚记录


In [7]:
# 年度偏移：2024 年为样本期内最晚记录。
show_svg(make_delay_line_svg(summary))

官方红叶日相对 12/5 的偏移 -5 +0 +5 +10 +15 2017 -4d 2024 +15d 2025 +5d 2010 2013 2016 2019 2022 2025 正值=比平年晚；2021 缺测未画点

### 4.2 散点回归：11 月均温越高，官方红叶日期越晚

该结果表明，10 月是否偏暖并不足以单独判断红叶日期是否显著推迟；11 月持续偏暖才是更直接的风险信号。


In [8]:
# 散点回归：11 月均温越高，官方红叶日期越晚。
show_svg(make_nov_scatter_svg(summary))

11月均温 vs 官方红叶延迟 -5 +0 +5 +10 +15 11°C 12°C 13°C 14°C 15°C 2015 2017 2024 2025 线性斜率约 3.0 天/°C；Pearson r≈0.714 横轴=11月均温；纵轴=相对 12/5 延迟天数

### 4.3 指标比较：10—11 月整体均温与 11 月均温解释力较强


In [9]:
# 指标比较：10—11 月整体均温与 11 月均温解释力较强。
show_svg(make_correlation_bar_svg(corr_by_metric))

各气温指标与红叶延迟的相关性 10-11月均温 +0.729 11月均温 +0.714 11/1-12/10均温 +0.709 ≤10°C冷天数 -0.644 12月均温 +0.393 10月均温 +0.309 红色=越暖越晚；蓝色=冷天越多越早

## 5. 结论与日期选择推论

统计结果支持以下判断：

- **常规年份：** 核心赏枫窗口可设为 11月28日—12月10日。
- **11 月显著偏暖：** 窗口应后移至 12月3日—12月14日。
- **11 月明显偏冷：** 窗口可提前至 11月22日—12月5日。
- **10 月偏暖但 11 月转冷：** 不宜仅凭 10 月气温判定红叶将显著推迟；2025 年属于这一类型。
- **10 月与 11 月均持续偏暖：** 需关注类似 2024 年的大幅推迟风险。

上述推论使用的是京都官方物候日期，而非任一寺社的现场“见顷”日期。高雄、大原、贵船、鞍马等北部或高海拔区域通常早于市区；清水寺、东福寺、下鸭神社等低海拔或市区点位可能更晚。


In [10]:
print("研究结论:")
print("1. 10 月均温单独解释力有限，不宜作为判断红叶显著推迟的唯一依据。")
print("2. 11 月均温是更直接的温度信号：每 +1°C，官方红叶日期约推迟 3 天。")
print("3. 10—11 月整体偏暖是样本中解释力最高的温度指标：每 +1°C，约推迟 4.4 天。")
print("4. 11/1—12/10 的 ≤10°C 冷日越多，红叶推进越快。")
print("5. 日期选择上，常规窗口为 11月28日—12月10日；若 11 月显著偏暖，可后移至 12月3日—12月14日。")

研究结论:
1. 10 月均温单独解释力有限，不宜作为判断红叶显著推迟的唯一依据。
2. 11 月均温是更直接的温度信号：每 +1°C，官方红叶日期约推迟 3 天。
3. 10—11 月整体偏暖是样本中解释力最高的温度指标：每 +1°C，约推迟 4.4 天。
4. 11/1—12/10 的 ≤10°C 冷日越多，红叶推进越快。
5. 日期选择上，常规窗口为 11月28日—12月10日；若 11 月显著偏暖，可后移至 12月3日—12月14日。
